# CPG Training: Fine-tuning T5-base Paraphraser on PAWS + QQP

This notebook fine-tunes `humarin/chatgpt_paraphraser_on_T5_base` (220M params) on:
- **PAWS** (labeled_final, label=1 pairs) — adversarial paraphrase pairs
- **QQP** (from GLUE, label=1 pairs) — Quora duplicate question pairs

**Runtime:** Google Colab with T4 GPU (free tier)

**Why this base model?** Raw t5-small produces near-copies (Self-BLEU=94). This model
is already trained on ChatGPT-generated paraphrases and produces diverse rewrites.
Fine-tuning further on PAWS+QQP customizes it for our use case.

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate sentencepiece torch

In [ ]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Load Base Model and Tokenizer

In [ ]:
MODEL_NAME = "humarin/chatgpt_paraphraser_on_T5_base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 2. Load and Prepare Training Data

We use positive pairs (label=1) from both datasets:
- PAWS: sentence pairs that are paraphrases
- QQP: Quora question pairs that are duplicates

In [ ]:
# Load PAWS — labeled_final split, label=1 (paraphrase pairs)
paws = load_dataset("paws", "labeled_final", split="train")
paws = paws.filter(lambda x: x["label"] == 1)
print(f"PAWS positive pairs: {len(paws)}")

# Rename columns to standard format
paws = paws.rename_column("sentence1", "input_text")
paws = paws.rename_column("sentence2", "target_text")
paws = paws.remove_columns(["label", "id"])

In [ ]:
# Load QQP from GLUE — label=1 (duplicate question pairs)
qqp = load_dataset("glue", "qqp", split="train")
qqp = qqp.filter(lambda x: x["label"] == 1)
print(f"QQP positive pairs: {len(qqp)}")

# Rename columns
qqp = qqp.rename_column("question1", "input_text")
qqp = qqp.rename_column("question2", "target_text")
qqp = qqp.remove_columns(["label", "idx"])

In [ ]:
# Combine datasets
# Use a subset for faster training (full dataset is large)
MAX_SAMPLES = 50000  # 50K total — enough for fine-tuning, fits in Colab time

paws_subset = paws.shuffle(seed=42).select(range(min(len(paws), MAX_SAMPLES // 2)))
qqp_subset = qqp.shuffle(seed=42).select(range(min(len(qqp), MAX_SAMPLES // 2)))

combined = concatenate_datasets([paws_subset, qqp_subset]).shuffle(seed=42)
print(f"Combined training set: {len(combined)} pairs")
print(f"Sample: {combined[0]}")

## 3. Tokenize the Dataset

In [ ]:
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 256

def tokenize_function(examples):
    """Tokenize input-target pairs with 'paraphrase: ' prefix."""
    inputs = [f"paraphrase: {text}" for text in examples["input_text"]]
    targets = examples["target_text"]

    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LENGTH, truncation=True, padding=False
    )

    labels = tokenizer(
        targets, max_length=MAX_TARGET_LENGTH, truncation=True, padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = combined.map(
    tokenize_function,
    batched=True,
    remove_columns=combined.column_names,
    desc="Tokenizing",
)

# Train/val split
split = tokenized.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## 4. Training Configuration

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./cpg-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Trainer ready. Starting fine-tuning...")

In [ ]:
# Train!
trainer.train()

## 5. Save the Fine-tuned Model

In [ ]:
# Save locally
SAVE_PATH = "./cpg-finetuned-final"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

In [ ]:
# Optional: Push to HuggingFace Hub
# from huggingface_hub import login
# login()
# trainer.push_to_hub("your-username/cpg-paraphraser")

## 6. Quick Test of Fine-tuned Model

In [ ]:
# Load fine-tuned model for inference
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

ft_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
ft_model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_PATH).to("cuda")
ft_model.eval()

test_sentence = "A cover letter is a formal document that accompanies your resume when you apply for a job."

inputs = ft_tokenizer(
    f"paraphrase: {test_sentence}",
    return_tensors="pt",
    max_length=256,
    truncation=True,
).to("cuda")

with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        do_sample=True,
        temperature=1.5,
        top_k=120,
        top_p=0.95,
        no_repeat_ngram_size=2,
        num_return_sequences=5,
        max_length=256,
    )

print("Input:", test_sentence)
print("\nGenerated candidates:")
for i, out in enumerate(outputs):
    decoded = ft_tokenizer.decode(out, skip_special_tokens=True)
    print(f"  [{i+1}] {decoded}")

In [ ]:
# Download the fine-tuned model (for local use)
!zip -r cpg-finetuned-final.zip cpg-finetuned-final/
from google.colab import files
files.download("cpg-finetuned-final.zip")